In [2]:
import sys 

sys.path.append('../')
from corpus.jsinV3_attn_cue_multi_source import jsinV3_attn_cue_multi_source
import src.audio_attention_transforms as aat
import src.audio_transforms as at


/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import numpy as np 
import torch 
import yaml

In [4]:
config_path = '../config/attentional_cue/attn_cue_match_target_speech_distractor_only.yaml'
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

In [5]:
config['data']

{'num_words': 794,
 'multi_distractor': True,
 'corpus': {'n_talkers': [1, 5],
  'with_audioset': False,
  'root': '/om2/user/msaddler/projects/ibmHearingAid/assets/data/datasets/JSIN_v3.00/nStim_20000/2000ms/rms_0.1/noiseSNR_-10_10/stimSR_20000/reverb_none/noise_all/JSIN_all_v3/subsets/'},
 'audio': {'rep_type': 'cochlea_filt',
  'rep_kwargs': {'sr': 20000,
   'env_sr': 8000,
   'n_channels': 40,
   'low_lim': 40,
   'use_pad': True,
   'rep_on_gpu': False,
   'env_extraction_type': 'Half-wave Rectification',
   'downsampling_type': 'TorchTransformsResample',
   'downsampling_kwargs': {'lowpass_filter_width': 64,
    'rolloff': 0.9475937167399596,
    'resampling_method': 'kaiser_window',
    'beta': 14.769656459379492}},
  'compression_type': 'coch_p3',
  'compression_kwargs': {'scale': 1,
   'offset': 1e-07,
   'clip_value': 5,
   'power': 0.3}},
 'loader': {'batch_size': 256}}

In [6]:
audio_transforms = aat.AudioCompose([
    aat.AudioToTensor(),
    aat.RMSNormalizeForegroundAndBackground(rms_level=0.1),
    aat.CombineWithRandomDBSNR(low_snr=config['noise_kwargs']['low_snr'], high_snr=config['noise_kwargs']['high_snr']),
    aat.RMSNormalizeMixtureAndMatchCueLevel(rms_level=0.1),
    aat.UnsqueezeAudio(dim=0),
])
# these transforms take foreground, background as input 
bg_combine_transforms = at.AudioCompose([
                at.AudioToTensor(),
                # at.CombineWithRandomDBSNR(low_snr=0, high_snr=0), # set distractors to same level for matched cue level training  
                # at.CombineWithRandomDBSNR(low_snr=config['noise_kwargs']['low_snr'], high_snr=config['noise_kwargs']['high_snr']),
                at.RMSNormalizeForegroundAndBackground(rms_level=0.1),
                # at.UnsqueezeAudio(dim=0),
            ])



In [7]:
coch_transform = aat.AudioToAudioRepresentation(**config['data']['audio'])


In [8]:
dataset = jsinV3_attn_cue_multi_source(**config['data']['corpus'],
                                       mode='train',
                                       noise_only=False,
                                       transform=[audio_transforms, bg_combine_transforms], demo=True)

In [14]:
foreground, background, signal, fg_cue, fg_target = dataset[211]

In [15]:
from IPython.display import Audio

In [16]:
# Audio(foreground, rate=20000, normalize=False)

In [17]:
Audio(fg_cue, rate=20000, normalize=False)

In [18]:
Audio(signal, rate=20000, normalize=False)



In [25]:
%matplotlib inline
import matplotlib.pyplot as plt

In [26]:
# plt.imshow(signal.squeeze().numpy(), aspect='auto')

In [ ]:
plt.imshow(fg_cue.squeeze().numpy(), aspect='auto')